# AIO Presence & Overlap Analysis — Part 2: Sources & Content
UGC sources, YouTube channel citations, top cited domains, and AIO content stats.
Standalone-runnable — loads and cleans its own data.

## Setup — load & clean data (shared preamble, standalone-runnable)

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from common import (
    load_data,
    set_plot_style,
    PALETTE,
)

sns.set_theme(style="whitegrid")
set_plot_style()
df = load_data()

## 6. UGC Sources: Organic vs AIO

How often do UGC platforms (YouTube, Reddit, Twitter/X, etc.) appear in organic top-10 results versus AIO citations?  
`ugc_share_organic` = fraction of organic top-10 domains that are UGC; `ugc_share_aio` = same for AIO sources (only for records where AIO appeared).

In [ ]:
UGC_DOMAINS = {
    "facebook.com",
    "instagram.com",
    "youtube.com",
    "reddit.com",
    "tiktok.com",
    "twitter.com",
    "x.com",
    "threads.com",
    "linkedin.com",
    "pinterest.com",
    "quora.com",
}


def ugc_share(domains_json):
    """Fraction of domains in a JSON-encoded list that belong to a UGC platform."""
    if not domains_json or (isinstance(domains_json, float)):
        return None
    try:
        domains = (
            json.loads(domains_json) if isinstance(domains_json, str) else domains_json
        )
    except Exception:
        return None
    if not domains:
        return None
    ugc = sum(
        1 for d in domains if any(d == u or d.endswith("." + u) for u in UGC_DOMAINS)
    )
    return ugc / len(domains)


df["ugc_share_organic"] = df["organic_domains"].apply(ugc_share)
df["ugc_share_aio"] = df["aio_domains"].apply(ugc_share)  # NaN where no AIO

n_aio_rows = df["has_ai_overview"].sum()
n_org_ugc = (df["ugc_share_organic"].fillna(0) > 0).sum()
n_aio_ugc = (df.loc[df["has_ai_overview"], "ugc_share_aio"].fillna(0) > 0).sum()

print(
    f"Queries with ≥1 UGC in organic  : {n_org_ugc}/{len(df)} ({n_org_ugc / len(df):.0%})"
)
print(
    f"Queries with ≥1 UGC in AIO      : {n_aio_ugc}/{n_aio_rows} ({n_aio_ugc / n_aio_rows:.0%})"
    if n_aio_rows
    else "No AIO data"
)
print(f"\nMean UGC share — organic : {df['ugc_share_organic'].mean():.1%}")
print(f"Mean UGC share — AIO     : {df['ugc_share_aio'].mean():.1%}")

In [ ]:
# ── 6a. Overall: mean UGC share — organic vs AIO ──────────────────────────────
org_mean = df["ugc_share_organic"].mean()
aio_mean = df["ugc_share_aio"].mean()

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(
    ["Organic top-10", "AIO sources"],
    [org_mean, aio_mean],
    color=["#3498db", "#e67e22"],
    width=0.5,
)
ax.bar_label(bars, labels=[f"{org_mean:.1%}", f"{aio_mean:.1%}"], padding=5)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_ylim(0, max(org_mean, aio_mean) * 1.5)
ax.set_ylabel("Mean UGC domain share")
ax.set_title("UGC platform share: AIO sources vs Organic top-10")
plt.tight_layout()
plt.show()

In [ ]:
# ── 6b. Per topic: grouped bar ────────────────────────────────────────────────
from pathlib import Path

topic_org = df.groupby("topic")["ugc_share_organic"].mean()
topic_aio = df.groupby("topic")["ugc_share_aio"].mean()
topic_ugc = pd.DataFrame({"Organic": topic_org, "AIO": topic_aio}).sort_index()

x = range(len(topic_ugc))
w = 0.35
fig, ax = plt.subplots(figsize=(max(7, len(topic_ugc) * 1.6), 5))
ax.bar(
    [i - w / 2 for i in x], topic_ugc["Organic"], w, label="Organic", color="#3498db"
)
ax.bar(
    [i + w / 2 for i in x], topic_ugc["AIO"].fillna(0), w, label="AIO", color="#e67e22"
)
ax.set_xticks(list(x))
ax.set_xticklabels(topic_ugc.index, rotation=30, ha="right", fontsize=14)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.tick_params(axis="y", labelsize=13)
ax.set_ylabel("Mean UGC domain share", fontsize=15)
ax.set_title("UGC share per topic: AIO vs Organic", fontsize=20)
ax.legend(fontsize=13)
plt.tight_layout()

FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURES_DIR / "ugc_share_per_topic.pdf", bbox_inches="tight")

plt.show()

In [ ]:
# ── 6c. Per stance: grouped bar ───────────────────────────────────────────────
df_st = df[df["stance"].notna()]

if df_st.empty:
    print("No stance data yet.")
else:
    order = ["Pro", "Neutral", "Con"]
    stance_org = df_st.groupby("stance")["ugc_share_organic"].mean().reindex(order)
    stance_aio = df_st.groupby("stance")["ugc_share_aio"].mean().reindex(order)

    x = range(len(order))
    w = 0.35
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(
        [i - w / 2 for i in x],
        stance_org.fillna(0),
        w,
        label="Organic",
        color="#3498db",
    )
    ax.bar(
        [i + w / 2 for i in x], stance_aio.fillna(0), w, label="AIO", color="#e67e22"
    )
    ax.set_xticks(list(x))
    ax.set_xticklabels(order)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_ylabel("Mean UGC domain share")
    ax.set_title("UGC share by stance: AIO vs Organic")
    ax.legend()
    plt.tight_layout()
    plt.show()

    print("\nPer-stance UGC share summary:")
    print(pd.DataFrame({"Organic": stance_org, "AIO": stance_aio}).round(3).to_string())

In [ ]:
from pathlib import Path


# ── 6d. Which UGC platforms appear, and where? ────────────────────────────────
def count_platforms(domains_series):
    counts = {u: 0 for u in sorted(UGC_DOMAINS)}
    for dj in domains_series.dropna():
        try:
            domains = json.loads(dj) if isinstance(dj, str) else dj
        except Exception:
            continue
        for d in domains:
            for u in UGC_DOMAINS:
                if d == u or d.endswith("." + u):
                    counts[u] += 1
                    break
    return pd.Series(counts)


n_total = len(df)
n_with_aio = int(df["has_ai_overview"].sum())

org_counts = count_platforms(df["organic_domains"])
aio_counts = count_platforms(df.loc[df["has_ai_overview"], "aio_domains"])

platform_df = pd.DataFrame(
    {
        "Organic (per 100 queries)": org_counts / n_total * 100,
        "AIO (per 100 AIO queries)": aio_counts / n_with_aio * 100 if n_with_aio else 0,
    }
)
platform_df = platform_df[(platform_df > 0).any(axis=1)].sort_values(
    "Organic (per 100 queries)", ascending=False
)

if platform_df.empty:
    print("No UGC platforms found in the current data.")
else:
    x = range(len(platform_df))
    w = 0.35
    fig, ax = plt.subplots(figsize=(max(7, len(platform_df) * 1.3), 5))
    ax.bar(
        [i - w / 2 for i in x],
        platform_df["Organic (per 100 queries)"],
        w,
        label="Organic",
        color="#3498db",
    )
    ax.bar(
        [i + w / 2 for i in x],
        platform_df["AIO (per 100 AIO queries)"],
        w,
        label="AIO",
        color="#e67e22",
    )
    ax.set_xticks(list(x))
    ax.set_xticklabels(platform_df.index, rotation=30, ha="right")
    ax.set_ylabel("Appearances per 100 queries")
    ax.set_title("UGC platform frequency: AIO vs Organic")
    ax.legend()
    plt.tight_layout()

    FIGURES_DIR = Path("figures")
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURES_DIR / "ugc_platform_frequency.pdf", bbox_inches="tight")

    plt.show()

    print(
        f"Normalised by: organic → {n_total} queries | AIO → {n_with_aio} queries with AIO"
    )
    display(platform_df.round(2))

### 6e. YouTube deep-dive: channel vs video citations

YouTube URLs encode the source type in their path:
- `/@handle` / `/user/name` / `/c/name` → **named channel** (identifiable)
- `/channel/UCxxxxxx` → opaque channel ID
- `/watch?v=xxx` / `/shorts/xxx` → **video only** (channel unknown from URL; check title)

For video-only URLs the `title` field from AIO sources often contains the channel name.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from config import YOUTUBE_API_KEY
import re
import requests as _req
from urllib.parse import urlparse, parse_qs

if not YOUTUBE_API_KEY:
    print("⚠️  YOUTUBE_API_KEY not found in .env")
else:
    print("✅ YouTube API key loaded")


# ── URL parser ────────────────────────────────────────────────────────────────
def parse_youtube_url(url):
    """Classify a YouTube URL; return yt_type, yt_name, video_id, channel_id_raw."""
    empty = {
        "yt_type": "invalid",
        "yt_name": None,
        "video_id": None,
        "channel_id_raw": None,
    }
    try:
        parsed = urlparse(url)
        path = parsed.path.rstrip("/")
    except Exception:
        return empty

    base = {"video_id": None, "channel_id_raw": None, "yt_name": None}

    for pattern, typ in [
        (r"^/@([^/]+)", "handle"),
        (r"^/user/([^/]+)", "username"),
        (r"^/c/([^/]+)", "custom"),
    ]:
        m = re.match(pattern, path)
        if m:
            return {**base, "yt_type": typ, "yt_name": m.group(1)}

    m = re.match(r"^/channel/(UC[A-Za-z0-9_-]+)", path)
    if m:
        return {**base, "yt_type": "channel_id", "channel_id_raw": m.group(1)}

    m = re.match(r"^/shorts/([A-Za-z0-9_-]+)", path)
    if m:
        return {**base, "yt_type": "short", "video_id": m.group(1)}

    qs = parse_qs(parsed.query)
    if "v" in qs:
        return {**base, "yt_type": "video", "video_id": qs["v"][0]}
    if "/watch" in path:
        return {**base, "yt_type": "video"}

    return {**base, "yt_type": "other", "yt_name": path}


# ── YouTube Data API v3 helpers ───────────────────────────────────────────────
def _yt_batch(endpoint, id_param, ids, api_key):
    """Batch YouTube API call (max 50 IDs per request). Returns {id: snippet}."""
    results = {}
    for i in range(0, len(ids), 50):
        batch = [x for x in ids[i : i + 50] if x]
        if not batch:
            continue
        try:
            resp = _req.get(
                f"https://www.googleapis.com/youtube/v3/{endpoint}",
                params={"part": "snippet", id_param: ",".join(batch), "key": api_key},
                timeout=15,
            )
            resp.raise_for_status()
            for item in resp.json().get("items", []):
                results[item["id"]] = item.get("snippet", {})
        except Exception as e:
            print(f"  YouTube API error ({endpoint}): {e}")
    return results


# ── Build raw df_yt ───────────────────────────────────────────────────────────
yt_rows = []
for _, row in df[df["has_ai_overview"]].iterrows():
    aio_raw = row.get("aio_sources")
    if not aio_raw or isinstance(aio_raw, float):
        continue
    try:
        sources = json.loads(aio_raw) if isinstance(aio_raw, str) else aio_raw
    except Exception:
        continue
    for src in sources:
        link = src.get("link", "")
        title = src.get("title", "")
        if "youtube.com" not in link and "youtu.be" not in link:
            continue
        yt_rows.append(
            {
                "query": row["query"],
                "topic": row["topic"],
                "stance": row.get("stance"),
                "title": title,
                "link": link,
                **parse_youtube_url(link),
            }
        )

df_yt = pd.DataFrame(yt_rows)
print(f"\nYouTube citations found: {len(df_yt)}")
if df_yt.empty:
    print("No YouTube citations in the current data.")
else:
    print("URL type breakdown (before API):")
    print(df_yt["yt_type"].value_counts().to_string())


# ── YouTube API enrichment ────────────────────────────────────────────────────
if not df_yt.empty and YOUTUBE_API_KEY:
    video_ids = df_yt.loc[df_yt["video_id"].notna(), "video_id"].unique().tolist()
    chan_ids = (
        df_yt.loc[df_yt["channel_id_raw"].notna(), "channel_id_raw"].unique().tolist()
    )

    vid_snippets = {}
    if video_ids:
        print(f"  → Fetching {len(video_ids)} video(s) from YouTube API…")
        vid_snippets = _yt_batch("videos", "id", video_ids, YOUTUBE_API_KEY)
        print(f"    resolved {len(vid_snippets)}/{len(video_ids)}")

    chan_snippets = {}
    if chan_ids:
        print(f"  → Fetching {len(chan_ids)} channel(s) from YouTube API…")
        chan_snippets = _yt_batch("channels", "id", chan_ids, YOUTUBE_API_KEY)
        print(f"    resolved {len(chan_snippets)}/{len(chan_ids)}")

    def _enrich(row):
        if row["video_id"] and row["video_id"] in vid_snippets:
            s = vid_snippets[row["video_id"]]
            return pd.Series(
                {
                    "channel_title": s.get("channelTitle"),
                    "channel_id": s.get("channelId"),
                    "api_resolved": True,
                }
            )
        if row["channel_id_raw"] and row["channel_id_raw"] in chan_snippets:
            s = chan_snippets[row["channel_id_raw"]]
            return pd.Series(
                {
                    "channel_title": s.get("title"),
                    "channel_id": row["channel_id_raw"],
                    "api_resolved": True,
                }
            )
        return pd.Series(
            {
                "channel_title": row["yt_name"],
                "channel_id": row.get("channel_id_raw"),
                "api_resolved": False,
            }
        )

    df_yt[["channel_title", "channel_id", "api_resolved"]] = df_yt.apply(
        _enrich, axis=1
    )
else:
    df_yt["channel_title"] = df_yt.get("yt_name")
    df_yt["channel_id"] = df_yt.get("channel_id_raw")
    df_yt["api_resolved"] = False

df_yt["channel_label"] = df_yt["channel_title"].fillna("(unknown)")

resolved = df_yt["api_resolved"].sum()
print(f"\nAPI-resolved channel info: {resolved}/{len(df_yt)} citations")
print(
    f"Channel label known      : {(df_yt['channel_label'] != '(unknown)').sum()}/{len(df_yt)}"
)


# ── URL-type pie ──────────────────────────────────────────────────────────────
if not df_yt.empty:
    type_counts = df_yt["yt_type"].value_counts()
    TYPE_COLORS = {
        "handle": "#27ae60",
        "username": "#2ecc71",
        "custom": "#82e0aa",
        "channel_id": "#f39c12",
        "video": "#e74c3c",
        "short": "#c0392b",
        "other": "#95a5a6",
        "invalid": "#7f8c8d",
    }
    colors = [TYPE_COLORS.get(t, "#bdc3c7") for t in type_counts.index]
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.pie(
        type_counts.values,
        labels=type_counts.index,
        colors=colors,
        autopct="%1.0f%%",
        startangle=140,
        wedgeprops={"linewidth": 1, "edgecolor": "white"},
    )
    ax.set_title("YouTube citations in AIO — URL type")
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Top channels cited (URL name + title-derived) ─────────────────────────────
if not df_yt.empty:
    n_unique_channels = df_yt.loc[
        df_yt["channel_label"] != "(unknown)", "channel_label"
    ].nunique()
    n_unknown = (df_yt["channel_label"] == "(unknown)").sum()
    n_name_only = (
        (df_yt["channel_label"] != "(unknown)") & ~df_yt["api_resolved"]
    ).sum()
    print(f"Total unique YouTube channels cited: {n_unique_channels}")
    print(
        f"Channel confirmed via YouTube API  : {df_yt['api_resolved'].sum()}/{len(df_yt)}"
    )
    print(f"Channel name from URL only (unconfirmed) : {n_name_only}/{len(df_yt)}")
    print(
        f"Citations with no channel identifiable   : {n_unknown}/{len(df_yt)} ({n_unknown / len(df_yt):.1%})"
    )

    top_channels = (
        df_yt[df_yt["channel_label"] != "(unknown)"]["channel_label"]
        .value_counts()
        .head(20)
    )

    CHANNEL_CATEGORIES = {
        "Giorgia Meloni News": "Right-wing party in power",
        "Fratelli d'Italia": "Right-wing party in power",
        "Il Fatto Quotidiano": "Daily newspaper (paper+online)",
        "La Repubblica": "Daily newspaper (paper+online)",
        "Il Sole 24 ORE": "Daily newspaper (paper+online)",
        "TG2000": "Television channel",
        "Rai": "Television channel",
        "La7 Attualità": "Television channel",
        "Fanpage.it": "Daily newspaper (online)",
        "Altalex News": "Daily newspaper (online)",
        "Adnkronos": "Daily newspaper (online)",
        "euronews (in Italiano)": "Daily newspaper (online)",
    }
    CATEGORY_COLORS = {
        "Right-wing party in power": "#39b0ff",
        "Daily newspaper (paper+online)": "#145b32",
        "Daily newspaper (online)": "#58cc2e",
        "Television channel": "#8e44ad",
    }
    DEFAULT_COLOR = "#d2ddddb2"

    if not top_channels.empty:
        bar_colors = [
            CATEGORY_COLORS.get(CHANNEL_CATEGORIES.get(ch), DEFAULT_COLOR)
            for ch in top_channels.index[::-1]
        ]
        fig, ax = plt.subplots(figsize=(10, max(4, len(top_channels) * 0.42)))
        ax.barh(
            top_channels.index[::-1],
            top_channels.values[::-1],
            color=bar_colors,
            alpha=0.85,
            edgecolor="white",
        )
        for i, v in enumerate(top_channels.values[::-1]):
            ax.text(v + 0.05, i, str(v), va="center", fontsize=9)
        ax.set_xlabel("Citations in AIO")
        ax.set_title("Top YouTube channels cited by Google AIO")

        legend_handles = [
            plt.Rectangle((0, 0), 1, 1, color=DEFAULT_COLOR, alpha=0.85, label="Other"),
        ] + [
            plt.Rectangle((0, 0), 1, 1, color=color, alpha=0.85, label=label)
            for label, color in CATEGORY_COLORS.items()
        ]
        ax.legend(handles=legend_handles, loc="lower right", fontsize=9)

        plt.tight_layout()
        plt.show()
    else:
        print("No named channels found — all citations are video-only URLs.")

## 7. Top Cited Domains: AIO vs Organic

Absolute citation counts for every domain across all records.  
Each row in `organic_domains` / `aio_domains` is a JSON list; we explode them and count appearances.  
(Repeated queries count multiple times — we also show a deduplicated view.)

In [ ]:
from pathlib import Path


def explode_domains(series, label):
    """Return a flat Series of domain strings from a column of JSON lists."""
    rows = []
    for dj in series.dropna():
        try:
            domains = json.loads(dj) if isinstance(dj, str) else dj
            rows.extend(d for d in domains if d)
        except Exception:
            pass
    return pd.Series(rows, name=label)


TOP_N = 25

# ── Raw counts (all records, including repeated queries) ──────────────────────
aio_domains_flat = explode_domains(df.loc[df["has_ai_overview"], "aio_domains"], "aio")
org_domains_flat = explode_domains(df["organic_domains"], "organic")

top_aio = aio_domains_flat.value_counts().head(TOP_N)
top_org = org_domains_flat.value_counts().head(TOP_N)

print(f"Unique domains in AIO     : {aio_domains_flat.nunique()}")
print(f"Unique domains in Organic : {org_domains_flat.nunique()}")
print(f"Total AIO citations       : {len(aio_domains_flat)}")
print(f"Total Organic citations   : {len(org_domains_flat)}")

fig, (ax_aio, ax_org) = plt.subplots(1, 2, figsize=(14, 6))

# AIO panel
ax_aio.barh(top_aio.index[::-1], top_aio.values[::-1], color="#e67e22")
ax_aio.set_xlabel("Appearances (raw)")
ax_aio.set_title(f"Top {TOP_N} domains — AIO sources")
for i, v in enumerate(top_aio.values[::-1]):
    ax_aio.text(v + 0.3, i, str(v), va="center", fontsize=8)

# Organic panel
ax_org.barh(top_org.index[::-1], top_org.values[::-1], color="#3498db")
ax_org.set_xlabel("Appearances (raw)")
ax_org.set_title(f"Top {TOP_N} domains — Organic top-10")
for i, v in enumerate(top_org.values[::-1]):
    ax_org.text(v + 0.3, i, str(v), va="center", fontsize=8)

plt.tight_layout()

FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURES_DIR / "top_domains_raw.pdf", bbox_inches="tight")

plt.show()

In [ ]:
# ── 7-bis. Top domains, linked: independent rankings, crossing flow ──────────
# Same idea as the entity-level flow chart in section 7a, but at the raw
# domain level (no YouTube-channel merging): AIO and Organic are each ranked
# independently, node thickness ∝ citation count, and a gradient ribbon
# (AIO orange → Organic blue) crosses the middle to link a domain's AIO node
# to its Organic node wherever it appears in both top lists. Labels sit on
# the inner edge, facing the ribbons.
import numpy as np
from matplotlib.collections import PolyCollection
from matplotlib.colors import to_rgb
from matplotlib.patches import Rectangle, Patch

SANKEY_TOP_N = 15
SEG_GAP = 0.3  # vertical gap between stacked node segments
NODE_W = 3.0  # horizontal thickness of each node column
GAP_W = 22.0  # horizontal span given to the ribbons
COLOR_AIO, COLOR_ORG = "#e67e22", "#3498db"

left_counts = aio_domains_flat.value_counts().head(SANKEY_TOP_N)
right_counts = org_domains_flat.value_counts().head(SANKEY_TOP_N)


def stack_layout(counts):
    """Top-to-bottom y-ranges for a stack of node segments, largest on top."""
    y_top = 0.0
    layout = {}
    for domain, count in counts.items():
        y_bot = y_top - count
        layout[domain] = (y_top, y_bot)
        y_top = y_bot - SEG_GAP
    return layout, y_top + SEG_GAP  # bottom of the whole stack


left_layout, left_bottom = stack_layout(left_counts)
right_layout, right_bottom = stack_layout(right_counts)
left_offset = -left_bottom / 2  # center each stack on y = 0 independently
right_offset = -right_bottom / 2

x_left_outer, x_left_inner = -(GAP_W / 2 + NODE_W), -GAP_W / 2
x_right_inner, x_right_outer = GAP_W / 2, GAP_W / 2 + NODE_W


def blend(c0, c1, f):
    c0, c1 = np.array(to_rgb(c0)), np.array(to_rgb(c1))
    return tuple(c0 + (c1 - c0) * f)


def sankey_ribbon(
    ax, x0, x1, y0_top, y0_bot, y1_top, y1_bot, color0, color1, n=50, alpha=0.55
):
    t = np.linspace(0, 1, n)
    ease = 0.5 - 0.5 * np.cos(np.pi * t)
    xs = x0 + (x1 - x0) * t
    top = y0_top + (y1_top - y0_top) * ease
    bot = y0_bot + (y1_bot - y0_bot) * ease
    verts, colors = [], []
    for i in range(n - 1):
        verts.append(
            [
                (xs[i], bot[i]),
                (xs[i], top[i]),
                (xs[i + 1], top[i + 1]),
                (xs[i + 1], bot[i + 1]),
            ]
        )
        colors.append(blend(color0, color1, (ease[i] + ease[i + 1]) / 2))
    ax.add_collection(
        PolyCollection(
            verts, facecolors=colors, edgecolors="none", alpha=alpha, zorder=1
        )
    )


fig, ax = plt.subplots(figsize=(10, 7))

shared_domains = set(left_layout) & set(right_layout)
for domain in shared_domains:
    y0_top, y0_bot = left_layout[domain]
    y1_top, y1_bot = right_layout[domain]
    sankey_ribbon(
        ax,
        x_left_inner,
        x_right_inner,
        y0_top + left_offset,
        y0_bot + left_offset,
        y1_top + right_offset,
        y1_bot + right_offset,
        COLOR_AIO,
        COLOR_ORG,
    )

for domain, (y_top, y_bot) in left_layout.items():
    y_top, y_bot = y_top + left_offset, y_bot + left_offset
    ax.add_patch(
        Rectangle(
            (x_left_outer, y_bot), NODE_W, y_top - y_bot, color=COLOR_AIO, zorder=2
        )
    )
    ax.text(
        x_left_inner + 0.3,
        (y_top + y_bot) / 2,
        domain,
        ha="left",
        va="center",
        fontsize=7.5,
        zorder=3,
    )
    ax.text(
        x_left_outer - 0.3,
        (y_top + y_bot) / 2,
        f"{int(left_counts[domain])}",
        ha="right",
        va="center",
        fontsize=10,
        color="#555555",
        zorder=3,
    )

for domain, (y_top, y_bot) in right_layout.items():
    y_top, y_bot = y_top + right_offset, y_bot + right_offset
    ax.add_patch(
        Rectangle(
            (x_right_inner, y_bot), NODE_W, y_top - y_bot, color=COLOR_ORG, zorder=2
        )
    )
    ax.text(
        x_right_inner - 0.3,
        (y_top + y_bot) / 2,
        domain,
        ha="right",
        va="center",
        fontsize=7.5,
        zorder=3,
    )
    ax.text(
        x_right_outer + 0.3,
        (y_top + y_bot) / 2,
        f"{int(right_counts[domain])}",
        ha="left",
        va="center",
        fontsize=10,
        color="#555555",
        zorder=3,
    )

ax.set_xlim(x_left_outer - 4, x_right_outer + 4)
y_min = min(left_bottom + left_offset, right_bottom + right_offset)
y_max = max(left_offset, right_offset)
y_pad = (y_max - y_min) * 0.03
ax.set_ylim(y_min - y_pad, y_max + y_pad)
ax.axis("off")
ax.set_title(
    f"AIO (top {SANKEY_TOP_N}) vs Organic (top {SANKEY_TOP_N}) domains — independently ranked, linked where shared"
)

ax.legend(
    handles=[
        Patch(color=COLOR_AIO, label="AIO sources"),
        Patch(color=COLOR_ORG, label="Organic top-10"),
    ],
    loc="lower center",
    bbox_to_anchor=(0.5, -0.02),
    ncol=2,
    frameon=False,
)

plt.tight_layout()

FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURES_DIR / "top_domains_flow_crossing.pdf", bbox_inches="tight")

plt.show()

In [ ]:
# ── 7a-bis. Same as above, but YouTube broken out by channel; cross-platform
# entities (a media outlet's website + its YouTube channel) merged into one bar.
from pathlib import Path


def extract_yt_links(df_in, json_col):
    rows = []
    for _, row in df_in[["query", json_col]].dropna().iterrows():
        try:
            items = (
                json.loads(row[json_col])
                if isinstance(row[json_col], str)
                else row[json_col]
            )
        except Exception:
            continue
        for it in items:
            link = it.get("link", "")
            if "youtube.com" not in link and "youtu.be" not in link:
                continue
            rows.append(
                {"query": row["query"], "link": link, **parse_youtube_url(link)}
            )
    return pd.DataFrame(rows)


yt_aio_links = extract_yt_links(df[df["has_ai_overview"]], "aio_sources")
yt_org_links = extract_yt_links(df, "organic_json")

# reuse the video/channel snippets already fetched in 6e; only fetch what's new
combined_video_ids = (
    pd.concat(
        [
            yt_aio_links.get("video_id", pd.Series(dtype=object)),
            yt_org_links.get("video_id", pd.Series(dtype=object)),
        ]
    )
    .dropna()
    .unique()
    .tolist()
)
combined_chan_ids = (
    pd.concat(
        [
            yt_aio_links.get("channel_id_raw", pd.Series(dtype=object)),
            yt_org_links.get("channel_id_raw", pd.Series(dtype=object)),
        ]
    )
    .dropna()
    .unique()
    .tolist()
)

new_video_ids = [v for v in combined_video_ids if v not in vid_snippets]
new_chan_ids = [c for c in combined_chan_ids if c not in chan_snippets]
if new_video_ids:
    vid_snippets.update(_yt_batch("videos", "id", new_video_ids, YOUTUBE_API_KEY))
if new_chan_ids:
    chan_snippets.update(_yt_batch("channels", "id", new_chan_ids, YOUTUBE_API_KEY))


def _channel_label(row):
    if row.get("video_id") and row["video_id"] in vid_snippets:
        return vid_snippets[row["video_id"]].get("channelTitle")
    if row.get("channel_id_raw") and row["channel_id_raw"] in chan_snippets:
        return chan_snippets[row["channel_id_raw"]].get("title")
    return row.get("yt_name")


for yt_links in (yt_aio_links, yt_org_links):
    if not yt_links.empty:
        yt_links["channel_label"] = yt_links.apply(_channel_label, axis=1).fillna(
            "(unknown)"
        )

# ── Manual mapping: same media entity, website domain <-> YouTube channel name ──
ENTITY_MERGE = {
    "La7 Attualità": "La7",
    "TG La7": "La7",
    "Il Fatto Quotidiano": "Il Fatto Quotidiano",
    "Fanpage.it": "Fanpage.it",
    "La Repubblica": "La Repubblica",
    "Il Sole 24 ORE": "Il Sole 24 ORE",
    "Adnkronos": "Adnkronos",
    "Altalex News": "Altalex",
    "ANM Associazione Nazionale Magistrati": "ANM",
    "Corriere della Sera": "Corriere della Sera",
    "euronews (in Italiano)": "Euronews",
    "Rai": "Rai",
    "Sky tg24": "Sky TG24",
    "Vatican News - Italiano": "Vatican News",
    "Confindustria": "Confindustria",
}
DOMAIN_TO_ENTITY = {
    "la7.it": "La7",
    "ilfattoquotidiano.it": "Il Fatto Quotidiano",
    "fanpage.it": "Fanpage.it",
    "repubblica.it": "La Repubblica",
    "ilsole24ore.com": "Il Sole 24 ORE",
    "adnkronos.com": "Adnkronos",
    "altalex.com": "Altalex",
    "associazionemagistrati.it": "ANM",
    "corriere.it": "Corriere della Sera",
    "euronews.com": "Euronews",
    "rai.it": "Rai",
    "rainews.it": "Rai",
    "tg24.sky.it": "Sky TG24",
    "vaticannews.va": "Vatican News",
    "confindustria.it": "Confindustria",
}


def build_merged_counts(domains_flat, yt_links_df):
    counts = {}
    for domain, n in domains_flat.value_counts().items():
        if domain == "youtube.com":
            continue
        entity = DOMAIN_TO_ENTITY.get(domain, domain)
        counts[entity] = counts.get(entity, 0) + n
    if not yt_links_df.empty:
        channel_counts = yt_links_df.loc[
            yt_links_df["channel_label"] != "(unknown)", "channel_label"
        ].value_counts()
        for channel, n in channel_counts.items():
            entity = ENTITY_MERGE.get(channel, f"{channel} (YouTube)")
            counts[entity] = counts.get(entity, 0) + n
    return pd.Series(counts).sort_values(ascending=False)


top_aio_yt = build_merged_counts(aio_domains_flat, yt_aio_links).head(TOP_N)
top_org_yt = build_merged_counts(org_domains_flat, yt_org_links).head(TOP_N)

fig, (ax_aio, ax_org) = plt.subplots(1, 2, figsize=(14, 6))

ax_aio.barh(top_aio_yt.index[::-1], top_aio_yt.values[::-1], color="#e67e22")
ax_aio.set_xlabel("Appearances (raw)")
ax_aio.set_title(f"Top {TOP_N} sources — AIO (YouTube by channel, entities merged)")
for i, v in enumerate(top_aio_yt.values[::-1]):
    ax_aio.text(v + 0.3, i, str(v), va="center", fontsize=8)

ax_org.barh(top_org_yt.index[::-1], top_org_yt.values[::-1], color="#3498db")
ax_org.set_xlabel("Appearances (raw)")
ax_org.set_title(f"Top {TOP_N} sources — Organic (YouTube by channel, entities merged)")
for i, v in enumerate(top_org_yt.values[::-1]):
    ax_org.text(v + 0.3, i, str(v), va="center", fontsize=8)

plt.tight_layout()

FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURES_DIR / "top_domains_with_channels.pdf", bbox_inches="tight")

plt.show()

In [ ]:
# ── 7a-ter. Same entities, linked: AIO vs Organic on one axis (dumbbell) ──
# The two-panel view above sorts AIO and Organic independently, so the same
# entity can sit at a different height in each panel — hard to compare by eye.
# Here every entity gets one row, with its AIO and Organic count as two dots
# joined by a line, so the AIO-vs-organic gap per entity is directly visible.
from pathlib import Path

LINK_TOP_N = 15  # entities considered per side before taking the union

full_aio_counts = build_merged_counts(aio_domains_flat, yt_aio_links)
full_org_counts = build_merged_counts(org_domains_flat, yt_org_links)

entities = full_aio_counts.head(LINK_TOP_N).index.union(
    full_org_counts.head(LINK_TOP_N).index, sort=False
)

linked = pd.DataFrame(
    {
        "aio": full_aio_counts.reindex(entities).fillna(0),
        "org": full_org_counts.reindex(entities).fillna(0),
    }
)
linked["total"] = linked["aio"] + linked["org"]
# ascending so that, once plotted bottom-up (barh-style), the largest total lands at the top
linked = linked.sort_values("total", ascending=True)

COLOR_AIO, COLOR_ORG, COLOR_LINE = "#e67e22", "#3498db", "#c7c7c7"

fig, ax = plt.subplots(figsize=(8, max(4, 0.4 * len(linked))))
y = range(len(linked))

ax.hlines(y, linked["org"], linked["aio"], color=COLOR_LINE, linewidth=2, zorder=1)
ax.scatter(
    linked["org"],
    y,
    s=80,
    color=COLOR_ORG,
    edgecolors="white",
    linewidths=1.5,
    zorder=2,
    label="Organic top-10",
)
ax.scatter(
    linked["aio"],
    y,
    s=80,
    color=COLOR_AIO,
    edgecolors="white",
    linewidths=1.5,
    zorder=2,
    label="AIO sources",
)

for i, (_, row) in enumerate(linked.iterrows()):
    ax.annotate(
        f"{int(row['org'])}",
        (row["org"], i),
        xytext=(0, -10),
        textcoords="offset points",
        ha="center",
        va="top",
        fontsize=7,
        color="#555555",
    )
    ax.annotate(
        f"{int(row['aio'])}",
        (row["aio"], i),
        xytext=(0, 10),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=7,
        color="#555555",
    )

ax.set_yticks(list(y))
ax.set_yticklabels(linked.index)
ax.set_xlabel("Appearances (raw)")
ax.set_xlim(left=0)
ax.set_title(f"AIO vs Organic citations per entity (top {len(linked)}, linked)")
ax.yaxis.grid(False)
ax.xaxis.grid(True, color="#dddddd", linewidth=0.8)
ax.legend(loc="lower right", frameon=False)

plt.tight_layout()

FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURES_DIR / "top_sources_linked.pdf", bbox_inches="tight")

plt.show()

In [ ]:
# ── 7a-quater. Flow chart, version A: shared row order, no crossing ──────────
# One row per entity (sorted by total citations), AIO flowing in from the
# left and Organic flowing out to the right of a central label — rendered as
# tapered "flow" ribbons instead of flat bars. Because both sides use the
# same row order, nothing needs to cross: this version answers "how much
# AIO vs Organic per entity", read top to bottom.
from matplotlib.patches import Patch

pyramid = linked.sort_values("total", ascending=False)
n_rows = len(pyramid)
ROW_H = 1.0
ys = np.arange(n_rows)[::-1]  # largest total at the top


def flow_ribbon(ax, y_center, x0, x1, half_h0, half_h1, color, alpha=0.9, n=40):
    t = np.linspace(0, 1, n)
    ease = 0.5 - 0.5 * np.cos(
        np.pi * t
    )  # smoothstep: flat at both ends, no kink at the label
    xs = x0 + (x1 - x0) * t
    half_h = half_h0 + (half_h1 - half_h0) * ease
    ax.fill_between(
        xs,
        y_center - half_h,
        y_center + half_h,
        color=color,
        linewidth=0,
        alpha=alpha,
        zorder=2,
    )


fig, ax = plt.subplots(figsize=(9, max(4, 0.5 * n_rows)))

max_val = max(pyramid["aio"].max(), pyramid["org"].max())
label_pad = max_val * 0.06

for y, (entity, row) in zip(ys, pyramid.iterrows()):
    flow_ribbon(ax, y, 0, -row["aio"], ROW_H * 0.42, ROW_H * 0.16, COLOR_AIO)
    flow_ribbon(ax, y, 0, row["org"], ROW_H * 0.42, ROW_H * 0.16, COLOR_ORG)
    if row["aio"] > 0:
        ax.text(
            -row["aio"] - label_pad * 0.3,
            y,
            f"{int(row['aio'])}",
            ha="right",
            va="center",
            fontsize=7,
            color="#555555",
        )
    if row["org"] > 0:
        ax.text(
            row["org"] + label_pad * 0.3,
            y,
            f"{int(row['org'])}",
            ha="left",
            va="center",
            fontsize=7,
            color="#555555",
        )
    ax.text(
        0,
        y,
        entity,
        ha="center",
        va="center",
        fontsize=8,
        color="#222222",
        zorder=3,
        bbox=dict(
            boxstyle="round,pad=0.25",
            facecolor="white",
            edgecolor="#dddddd",
            linewidth=0.8,
        ),
    )

ax.set_xlim(
    -pyramid["aio"].max() * 1.25 - label_pad, pyramid["org"].max() * 1.25 + label_pad
)
ax.set_ylim(-1, n_rows)
ax.set_yticks([])
ax.set_xlabel("← AIO appearances          Organic appearances →")
ax.set_title(f"AIO vs Organic citations per entity (top {n_rows}, shared order)")
ax.axvline(0, color="#cccccc", linewidth=1, zorder=1)
for spine in ("top", "right", "left"):
    ax.spines[spine].set_visible(False)

ax.legend(
    handles=[
        Patch(color=COLOR_AIO, label="AIO sources"),
        Patch(color=COLOR_ORG, label="Organic top-10"),
    ],
    loc="lower center",
    bbox_to_anchor=(0.5, 1.02),
    ncol=2,
    frameon=False,
)

plt.tight_layout()

FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURES_DIR / "top_sources_flow_shared_order.pdf", bbox_inches="tight")

plt.show()

In [ ]:
# ── 7a-quinquies. Flow chart, version B: independent rankings, crossing ──────
# AIO and Organic are each ranked independently (as in 7a-bis), so the same
# entity can land at a different height on each side. Node thickness ∝ count;
# a gradient ribbon (AIO orange → Organic blue) crosses the middle gap to
# link the same entity's AIO node to its Organic node wherever it appears in
# both top lists — inspired by a Sankey diagram, but with labels moved to the
# inner edge (facing the ribbons) instead of the outer edge.
import numpy as np
from matplotlib.patches import Rectangle, Patch

SANKEY_TOP_N = 15
SEG_GAP = 0.3  # vertical gap between stacked node segments
NODE_W = 3.0  # horizontal thickness of each node column
GAP_W = 22.0  # horizontal span given to the ribbons

left_counts = full_aio_counts.head(SANKEY_TOP_N)
right_counts = full_org_counts.head(SANKEY_TOP_N)


def stack_layout(counts):
    """Top-to-bottom y-ranges for a stack of node segments, largest on top."""
    y_top = 0.0
    layout = {}
    for entity, count in counts.items():
        y_bot = y_top - count
        layout[entity] = (y_top, y_bot)
        y_top = y_bot - SEG_GAP
    return layout, y_top + SEG_GAP  # bottom of the whole stack


left_layout, left_bottom = stack_layout(left_counts)
right_layout, right_bottom = stack_layout(right_counts)
left_offset = -left_bottom / 2  # center each stack on y = 0 independently
right_offset = -right_bottom / 2

x_left_outer, x_left_inner = -(GAP_W / 2 + NODE_W), -GAP_W / 2
x_right_inner, x_right_outer = GAP_W / 2, GAP_W / 2 + NODE_W


# blend() is defined in the "7-bis" cell above (§7) and reused here — the
# notebook must be run top-to-bottom for this cell to see it.


def sankey_ribbon(
    ax, x0, x1, y0_top, y0_bot, y1_top, y1_bot, color0, color1, n=50, alpha=0.55
):
    t = np.linspace(0, 1, n)
    ease = 0.5 - 0.5 * np.cos(np.pi * t)
    xs = x0 + (x1 - x0) * t
    top = y0_top + (y1_top - y0_top) * ease
    bot = y0_bot + (y1_bot - y0_bot) * ease
    verts, colors = [], []
    for i in range(n - 1):
        verts.append(
            [
                (xs[i], bot[i]),
                (xs[i], top[i]),
                (xs[i + 1], top[i + 1]),
                (xs[i + 1], bot[i + 1]),
            ]
        )
        colors.append(blend(color0, color1, (ease[i] + ease[i + 1]) / 2))
    ax.add_collection(
        PolyCollection(
            verts, facecolors=colors, edgecolors="none", alpha=alpha, zorder=1
        )
    )


fig, ax = plt.subplots(figsize=(10, 7))

shared_entities = set(left_layout) & set(right_layout)
for entity in shared_entities:
    y0_top, y0_bot = left_layout[entity]
    y1_top, y1_bot = right_layout[entity]
    sankey_ribbon(
        ax,
        x_left_inner,
        x_right_inner,
        y0_top + left_offset,
        y0_bot + left_offset,
        y1_top + right_offset,
        y1_bot + right_offset,
        COLOR_AIO,
        COLOR_ORG,
    )

for entity, (y_top, y_bot) in left_layout.items():
    y_top, y_bot = y_top + left_offset, y_bot + left_offset
    ax.add_patch(
        Rectangle(
            (x_left_outer, y_bot), NODE_W, y_top - y_bot, color=COLOR_AIO, zorder=2
        )
    )
    ax.text(
        x_left_inner + 0.3,
        (y_top + y_bot) / 2,
        entity,
        ha="left",
        va="center",
        fontsize=7.5,
        zorder=3,
    )
    ax.text(
        x_left_outer - 0.3,
        (y_top + y_bot) / 2,
        f"{int(left_counts[entity])}",
        ha="right",
        va="center",
        fontsize=10,
        color="#555555",
        zorder=3,
    )

for entity, (y_top, y_bot) in right_layout.items():
    y_top, y_bot = y_top + right_offset, y_bot + right_offset
    ax.add_patch(
        Rectangle(
            (x_right_inner, y_bot), NODE_W, y_top - y_bot, color=COLOR_ORG, zorder=2
        )
    )
    ax.text(
        x_right_inner - 0.3,
        (y_top + y_bot) / 2,
        entity,
        ha="right",
        va="center",
        fontsize=7.5,
        zorder=3,
    )
    ax.text(
        x_right_outer + 0.3,
        (y_top + y_bot) / 2,
        f"{int(right_counts[entity])}",
        ha="left",
        va="center",
        fontsize=10,
        color="#555555",
        zorder=3,
    )

ax.set_xlim(x_left_outer - 4, x_right_outer + 4)
y_min = min(left_bottom + left_offset, right_bottom + right_offset)
y_max = max(left_offset, right_offset)
y_pad = (y_max - y_min) * 0.03
ax.set_ylim(y_min - y_pad, y_max + y_pad)
ax.axis("off")
ax.set_title(
    f"AIO (top {SANKEY_TOP_N}) vs Organic (top {SANKEY_TOP_N}) — independently ranked, linked where shared"
)

ax.legend(
    handles=[
        Patch(color=COLOR_AIO, label="AIO sources"),
        Patch(color=COLOR_ORG, label="Organic top-10"),
    ],
    loc="lower center",
    bbox_to_anchor=(0.5, -0.02),
    ncol=2,
    frameon=False,
)

plt.tight_layout()

FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURES_DIR / "top_sources_flow_crossing.pdf", bbox_inches="tight")

plt.show()

In [ ]:
# ── Where does facebook.com appear most as an AIO source? (topic × stance) ────
def domain_present(domains_json, domain):
    if not domains_json or isinstance(domains_json, float):
        return False
    try:
        domains = (
            json.loads(domains_json) if isinstance(domains_json, str) else domains_json
        )
    except Exception:
        return False
    return domain in domains


TARGET_DOMAIN = "facebook.com"

df_aio = df[df["has_ai_overview"] & df["stance"].notna()].copy()
df_aio["has_fb"] = df_aio["aio_domains"].apply(
    lambda d: domain_present(d, TARGET_DOMAIN)
)

STANCE_ORDER = ["Pro", "Neutral", "Con"]
STANCE_LABELS = {"Pro": "Pros", "Neutral": "Neutral", "Con": "Cons"}

fb_stats = (
    df_aio.groupby(["topic", "stance"])["has_fb"]
    .agg(total="count", fb_present="sum")
    .assign(fb_rate=lambda x: x["fb_present"] / x["total"])
    .sort_values("fb_present", ascending=False)
)
display(fb_stats.style.format({"fb_rate": "{:.0%}"}))

fb_pivot = (
    fb_stats["fb_present"].unstack("stance").reindex(columns=STANCE_ORDER).fillna(0)
)
fb_pivot = fb_pivot.loc[fb_pivot.sum(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(10, 5))
fb_pivot.rename(columns=STANCE_LABELS).plot(
    kind="bar", ax=ax, color=[PALETTE[s] for s in STANCE_ORDER]
)
ax.set_ylabel(f"Queries citing {TARGET_DOMAIN} in AIO")
ax.set_xlabel("")
ax.set_title(f"AIO citations of {TARGET_DOMAIN} by topic × stance")
ax.legend(title="Stance")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# ── 7b. Deduplicated: each (query, domain) pair counted once ─────────────────
def explode_domains_dedup(df_in, domains_col, query_col="query"):
    rows = []
    for _, row in df_in[[query_col, domains_col]].dropna().iterrows():
        try:
            domains = (
                json.loads(row[domains_col])
                if isinstance(row[domains_col], str)
                else row[domains_col]
            )
            for d in set(domains):
                if d:
                    rows.append(d)
        except Exception:
            pass
    return pd.Series(rows)


top_aio_dd = (
    explode_domains_dedup(df[df["has_ai_overview"]], "aio_domains")
    .value_counts()
    .head(TOP_N)
)
top_org_dd = explode_domains_dedup(df, "organic_domains").value_counts().head(TOP_N)

fig, (ax_aio, ax_org) = plt.subplots(1, 2, figsize=(14, 6))

ax_aio.barh(top_aio_dd.index[::-1], top_aio_dd.values[::-1], color="#e67e22")
ax_aio.set_xlabel("Unique queries citing domain")
ax_aio.set_title(f"Top {TOP_N} domains — AIO (deduplicated)")
for i, v in enumerate(top_aio_dd.values[::-1]):
    ax_aio.text(v + 0.1, i, str(v), va="center", fontsize=8)

ax_org.barh(top_org_dd.index[::-1], top_org_dd.values[::-1], color="#3498db")
ax_org.set_xlabel("Unique queries citing domain")
ax_org.set_title(f"Top {TOP_N} domains — Organic (deduplicated)")
for i, v in enumerate(top_org_dd.values[::-1]):
    ax_org.text(v + 0.1, i, str(v), va="center", fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ── 7c. AIO rank vs Organic rank for shared domains ──────────────────────────
aio_ranks = (
    aio_domains_flat.value_counts()
    .rank(method="min", ascending=False)
    .rename("aio_rank")
)
org_ranks = (
    org_domains_flat.value_counts()
    .rank(method="min", ascending=False)
    .rename("org_rank")
)

shared = pd.concat([aio_ranks, org_ranks], axis=1).dropna()
shared["aio_count"] = aio_domains_flat.value_counts()
shared["org_count"] = org_domains_flat.value_counts()
shared = shared.sort_values("aio_count", ascending=False)

print(f"Domains appearing in both AIO and Organic: {len(shared)}")
display(shared.head(20).astype({"aio_rank": int, "org_rank": int}))

if len(shared) >= 3:
    fig, ax = plt.subplots(figsize=(7, 6))
    sc = ax.scatter(
        shared["org_rank"],
        shared["aio_rank"],
        s=shared["aio_count"] * 10,
        alpha=0.7,
        color="#9b59b6",
        edgecolors="white",
    )
    for domain, row in shared.head(12).iterrows():
        ax.annotate(
            domain,
            (row["org_rank"], row["aio_rank"]),
            fontsize=7,
            xytext=(4, 4),
            textcoords="offset points",
        )
    ax.set_xlabel("Organic rank (1 = most cited)")
    ax.set_ylabel("AIO rank (1 = most cited)")
    ax.set_title("Domain rank: AIO vs Organic\n(bubble size = AIO citation count)")
    ax.invert_xaxis()
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

In [ ]:
# ── 7d. Heatmap: domain citation rate per topic (% of queries) ────────────────
TOP_DOMAINS_HEAT = 20


def domain_topic_pct(df_in, domains_col, topic_col="topic"):
    """
    Returns a (domain × topic) DataFrame where each cell is the % of queries
    in that topic that cite the domain (deduplicated: one count per query).
    """
    n_per_topic = df_in.groupby(topic_col).size()
    rows = []
    for _, row in df_in[[topic_col, domains_col]].dropna().iterrows():
        try:
            doms = (
                json.loads(row[domains_col])
                if isinstance(row[domains_col], str)
                else row[domains_col]
            )
            for d in set(doms):
                if d:
                    rows.append({"topic": row[topic_col], "domain": d})
        except Exception:
            pass
    if not rows:
        return pd.DataFrame()
    flat = pd.DataFrame(rows)
    mat = (
        flat.groupby(["topic", "domain"]).size().unstack(fill_value=0)
    )  # topic × domain
    return (mat.div(n_per_topic, axis=0) * 100).T  # → domain × topic


aio_mat = domain_topic_pct(df[df["has_ai_overview"]], "aio_domains")
org_mat = domain_topic_pct(df, "organic_domains")

# Select top domains by their peak % across any topic
top_aio_doms = (
    aio_mat.max(axis=1).nlargest(TOP_DOMAINS_HEAT).index if not aio_mat.empty else []
)
top_org_doms = (
    org_mat.max(axis=1).nlargest(TOP_DOMAINS_HEAT).index if not org_mat.empty else []
)

n_topics = max(
    len(aio_mat.columns) if not aio_mat.empty else 1,
    len(org_mat.columns) if not org_mat.empty else 1,
)
fig_w = max(6, n_topics * 1.4)

# ── AIO heatmap ───────────────────────────────────────────────────────────────
if not aio_mat.empty and len(top_aio_doms):
    fig, ax = plt.subplots(figsize=(fig_w, max(4, len(top_aio_doms) * 0.45)))
    sns.heatmap(
        aio_mat.loc[top_aio_doms].round(0).astype(int),
        annot=True,
        fmt="d",
        cmap="YlOrRd",
        vmin=0,
        vmax=100,
        linewidths=0.4,
        linecolor="#eee",
        cbar_kws={"label": "% of AIO queries citing domain"},
        ax=ax,
    )
    ax.set_title("Domain citation rate per topic — AIO sources (%)")
    ax.set_xlabel("Topic")
    ax.set_ylabel("Domain")
    ax.tick_params(axis="x", rotation=30)
    ax.tick_params(axis="y", rotation=0)
    plt.tight_layout()
    plt.show()

# ── Organic heatmap ───────────────────────────────────────────────────────────
if not org_mat.empty and len(top_org_doms):
    fig, ax = plt.subplots(figsize=(fig_w, max(4, len(top_org_doms) * 0.45)))
    sns.heatmap(
        org_mat.loc[top_org_doms].round(0).astype(int),
        annot=True,
        fmt="d",
        cmap="Blues",
        vmin=0,
        vmax=100,
        linewidths=0.4,
        linecolor="#eee",
        cbar_kws={"label": "% of queries citing domain"},
        ax=ax,
    )
    ax.set_title("Domain citation rate per topic — Organic top-10 (%)")
    ax.set_xlabel("Topic")
    ax.set_ylabel("Domain")
    ax.tick_params(axis="x", rotation=30)
    ax.tick_params(axis="y", rotation=0)
    plt.tight_layout()
    plt.show()

## 8. AIO Content — General Stats

Focused only on records where Google returned an AI Overview.  
Metrics: number of cited sources, text length (characters & words), breakdowns by topic and stance.

In [ ]:
# ── 8. Setup: AIO-only slice + derived text metrics ──────────────────────────
df_a = df[df["has_ai_overview"]].copy()
df_a["aio_source_count"] = (
    pd.to_numeric(df_a["aio_source_count"], errors="coerce").fillna(0).astype(int)
)
df_a["aio_text_chars"] = df_a["aio_text"].str.len()
df_a["aio_text_words"] = df_a["aio_text"].str.split().str.len()

n_with_text = df_a["aio_text"].notna().sum()
n_no_text = df_a["aio_text"].isna().sum()

print(
    f"AIO records          : {len(df_a)} / {len(df)} total ({len(df_a) / len(df):.0%})"
)
print(f"  with text content  : {n_with_text}")
print(f"  empty AIO (no text): {n_no_text}")
print()

summary = (
    df_a[["aio_source_count", "aio_text_chars", "aio_text_words"]]
    .describe()
    .loc[["count", "mean", "std", "min", "25%", "50%", "75%", "max"]]
    .rename(
        columns={
            "aio_source_count": "sources cited",
            "aio_text_chars": "text length (chars)",
            "aio_text_words": "text length (words)",
        }
    )
    .round(1)
)
display(summary)

In [ ]:
# ── 8a. Distributions: source count & text word count ────────────────────────
from pathlib import Path

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Sources per AIO
ax = axes[0]
ax.hist(
    df_a["aio_source_count"],
    bins=range(0, df_a["aio_source_count"].max() + 2),
    color="#e67e22",
    edgecolor="white",
    align="left",
)
ax.axvline(
    df_a["aio_source_count"].mean(),
    color="black",
    linestyle="--",
    linewidth=1.2,
    label=f"mean {df_a['aio_source_count'].mean():.1f}",
)
ax.axvline(
    df_a["aio_source_count"].median(),
    color="grey",
    linestyle=":",
    linewidth=1.2,
    label=f"median {df_a['aio_source_count'].median():.0f}",
)
ax.set_xlabel("Sources cited per AIO", fontsize=14)
ax.set_ylabel("Number of queries", fontsize=14)
ax.set_title("Source count distribution", fontsize=16)
ax.tick_params(axis="both", labelsize=12)
ax.legend(fontsize=13)

# Text length — characters
ax = axes[1]
df_text = df_a["aio_text_chars"].dropna()
ax.hist(df_text, bins=20, color="#9b59b6", edgecolor="white")
ax.axvline(
    df_text.mean(),
    color="black",
    linestyle="--",
    linewidth=1.2,
    label=f"mean {df_text.mean():.0f}",
)
ax.axvline(
    df_text.median(),
    color="grey",
    linestyle=":",
    linewidth=1.2,
    label=f"median {df_text.median():.0f}",
)
ax.set_xlabel("AIO text length (characters)", fontsize=14)
ax.set_ylabel("Number of queries", fontsize=14)
ax.set_title("Text length — characters", fontsize=16)
ax.tick_params(axis="both", labelsize=12)
ax.legend(fontsize=13)

# Text length — words
ax = axes[2]
df_words = df_a["aio_text_words"].dropna()
ax.hist(df_words, bins=20, color="#1abc9c", edgecolor="white")
ax.axvline(
    df_words.mean(),
    color="black",
    linestyle="--",
    linewidth=1.2,
    label=f"mean {df_words.mean():.0f}",
)
ax.axvline(
    df_words.median(),
    color="grey",
    linestyle=":",
    linewidth=1.2,
    label=f"median {df_words.median():.0f}",
)
ax.set_xlabel("AIO text length (words)", fontsize=14)
ax.set_ylabel("Number of queries", fontsize=14)
ax.set_title("Text length — words", fontsize=16)
ax.tick_params(axis="both", labelsize=12)
ax.legend(fontsize=13)

plt.suptitle("AIO content distributions", y=1.04, fontsize=18, fontweight="bold")
plt.tight_layout()

FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURES_DIR / "aio_content_distributions.pdf", bbox_inches="tight")

plt.show()

In [ ]:
# ── 8b. Source count per topic ────────────────────────────────────────────────
topics_order = sorted(df_a["topic"].dropna().unique())

fig, axes = plt.subplots(1, 2, figsize=(max(10, len(topics_order) * 2.2), 5))

# Sources
ax = axes[0]
sns.boxplot(
    data=df_a,
    x="topic",
    y="aio_source_count",
    order=topics_order,
    color="#e67e22",
    width=0.5,
    ax=ax,
)
sns.stripplot(
    data=df_a,
    x="topic",
    y="aio_source_count",
    order=topics_order,
    color="black",
    size=3,
    alpha=0.5,
    jitter=True,
    ax=ax,
)
ax.set_xticklabels(topics_order, rotation=30, ha="right")
ax.set_ylabel("Sources cited")
ax.set_xlabel("")
ax.set_title("Sources cited per AIO — by topic")

# Word count
ax = axes[1]
df_a_text = df_a[df_a["aio_text_words"].notna()]
sns.boxplot(
    data=df_a_text,
    x="topic",
    y="aio_text_words",
    order=topics_order,
    color="#1abc9c",
    width=0.5,
    ax=ax,
)
sns.stripplot(
    data=df_a_text,
    x="topic",
    y="aio_text_words",
    order=topics_order,
    color="black",
    size=3,
    alpha=0.5,
    jitter=True,
    ax=ax,
)
ax.set_xticklabels(topics_order, rotation=30, ha="right")
ax.set_ylabel("Word count")
ax.set_xlabel("")
ax.set_title("AIO text length (words) — by topic")

plt.tight_layout()
plt.show()

print("\nMean sources cited per topic:")
print(df_a.groupby("topic")["aio_source_count"].describe().round(1).to_string())

In [ ]:
# ── 8c. Source count & text length per stance ─────────────────────────────────
df_a_st = df_a[df_a["stance"].notna()]

if df_a_st.empty:
    print("No stance data yet.")
else:
    stance_order = ["Pro", "Neutral", "Con"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    ax = axes[0]
    sns.boxplot(
        data=df_a_st,
        x="stance",
        y="aio_source_count",
        order=stance_order,
        palette=PALETTE,
        width=0.5,
        ax=ax,
    )
    sns.stripplot(
        data=df_a_st,
        x="stance",
        y="aio_source_count",
        order=stance_order,
        color="black",
        size=4,
        alpha=0.5,
        jitter=True,
        ax=ax,
    )
    ax.set_xlabel("Query stance")
    ax.set_ylabel("Sources cited")
    ax.set_title("Sources cited per AIO — by stance")

    ax = axes[1]
    df_a_st_text = df_a_st[df_a_st["aio_text_words"].notna()]
    sns.boxplot(
        data=df_a_st_text,
        x="stance",
        y="aio_text_words",
        order=stance_order,
        palette=PALETTE,
        width=0.5,
        ax=ax,
    )
    sns.stripplot(
        data=df_a_st_text,
        x="stance",
        y="aio_text_words",
        order=stance_order,
        color="black",
        size=4,
        alpha=0.5,
        jitter=True,
        ax=ax,
    )
    ax.set_xlabel("Query stance")
    ax.set_ylabel("Word count")
    ax.set_title("AIO text length (words) — by stance")

    plt.tight_layout()
    plt.show()

    print("\nSources cited — by stance:")
    print(
        df_a_st.groupby("stance")["aio_source_count"]
        .describe()
        .round(1)
        .reindex(stance_order)
        .to_string()
    )
    print("\nText word count — by stance:")
    print(
        df_a_st.groupby("stance")["aio_text_words"]
        .describe()
        .round(1)
        .reindex(stance_order)
        .to_string()
    )

In [ ]:
# ── 8d. Source count & text length per political leaning ──────────────────────
df_a_lean = df_a[df_a["pro_leaning"].notna()]

if df_a_lean.empty:
    print("No pro_leaning data yet.")
else:
    leaning_order = ["Left", "Right"]
    LEANING_COLORS = {"Left": "#c0392b", "Right": "#2471a3"}
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    ax = axes[0]
    sns.boxplot(
        data=df_a_lean,
        x="pro_leaning",
        y="aio_source_count",
        order=leaning_order,
        palette=LEANING_COLORS,
        width=0.5,
        ax=ax,
    )
    sns.stripplot(
        data=df_a_lean,
        x="pro_leaning",
        y="aio_source_count",
        order=leaning_order,
        color="black",
        size=4,
        alpha=0.5,
        jitter=True,
        ax=ax,
    )
    ax.set_xlabel("Political leaning")
    ax.set_ylabel("Sources cited")
    ax.set_title("Sources cited per AIO — by leaning")

    ax = axes[1]
    df_a_lean_text = df_a_lean[df_a_lean["aio_text_words"].notna()]
    sns.boxplot(
        data=df_a_lean_text,
        x="pro_leaning",
        y="aio_text_words",
        order=leaning_order,
        palette=LEANING_COLORS,
        width=0.5,
        ax=ax,
    )
    sns.stripplot(
        data=df_a_lean_text,
        x="pro_leaning",
        y="aio_text_words",
        order=leaning_order,
        color="black",
        size=4,
        alpha=0.5,
        jitter=True,
        ax=ax,
    )
    ax.set_xlabel("Political leaning")
    ax.set_ylabel("Word count")
    ax.set_title("AIO text length (words) — by leaning")

    plt.tight_layout()
    plt.show()

    print("\nSources cited — by leaning:")
    print(
        df_a_lean.groupby("pro_leaning")["aio_source_count"]
        .describe()
        .round(1)
        .reindex(leaning_order)
        .to_string()
    )
    print("\nText word count — by leaning:")
    print(
        df_a_lean.groupby("pro_leaning")["aio_text_words"]
        .describe()
        .round(1)
        .reindex(leaning_order)
        .to_string()
    )